In [ ]:
import time
import numpy as np
from submit import my_latent
from submit import my_latent_updated

In [ ]:
import numpy as np
from submit import my_latent, my_latent_updated

data_trn = np.loadtxt("public_trn.txt")
data_tst = np.loadtxt("public_tst.txt")
X, y = data_trn[:, :-1], data_trn[:, -1]
X_t, y_t = data_tst[:, :-1], data_tst[:, -1]

w, b, z = my_latent(X, y)
print(w.shape, b, z.shape, z.dtype)   # sanity check shapes/types

w2, b2, u2, a2 = my_latent_updated(X, y)
print(w2.shape, b2, u2.shape, a2)

(17,) -0.10473191754962397 (8000,) bool
(17,) -0.26345225090198293 (16,) 1.4097563881262396


In [ ]:
data_trn = np.loadtxt( "private_trn.txt" )
data_tst = np.loadtxt( "private_tst.txt" )

X = data_trn[ :, :-1 ]
y = data_trn[ :, -1 ]
X_t = data_tst[ :, :-1 ]
y_t = data_tst[ :, -1 ]

n_trials = 10

t_train1 = np.inf
a_train = 0.0
t_train2 = np.inf
a_test = 0.0

In [ ]:
def accuracy_score( y, pred ):
  return np.average( y == pred )

def predict( X, w, b ):
  pred = np.zeros( ( len( X ), ) )
  score = X.dot( w ) + b
  pred[ score > 0 ] = 1
  return pred

def apuf_features( X ):
  return np.flip( np.cumprod( np.flip( 1 - 2 * X, axis = 1 ), axis = 1 ), axis = 1 )

def insert( X, z, midpos ):
  return np.concatenate( [ X[ :, :midpos ], z[ :, np.newaxis ], X[ :, midpos: ] ], axis = 1 )

for t in range( n_trials ):
  # Get the model and latent variables
  tic = time.perf_counter()
  w, b, z = my_latent( X, y )
  toc = time.perf_counter()
  tt = toc - tic
  t_train1 = min( t_train1, tt )

  # Get train accuracy
  acc = accuracy_score( y, predict( apuf_features( insert( X, z, midpos = 8 ) ), w, b ) )
  a_train = max( a_train, acc )

  # Get the two models
  tic = time.perf_counter()
  w, b, u, a = my_latent_updated( X, y )
  toc = time.perf_counter()
  tt = toc - tic
  t_train2 = min( t_train2, tt )

  # Get test accuracy
  z_pred = predict( apuf_features( X_t ), u, a )
  acc = accuracy_score( y_t, predict( apuf_features( insert( X_t, z_pred, midpos = 8 ) ), w, b ) )
  a_test = max( a_test, acc )

In [ ]:
print( f"{t_train1},{a_train},{t_train2},{a_test}" )

0.05510446399966895,1.0,0.7815955850001046,0.995
